# Jing RAG Minimal 040326

This notebook implements a reproducible, local-only (no external LLM API) SciQ RAG minimal loop.


## 0_setup

- **Goal**: Initialize deterministic runtime, imports, and device fallback.
- **Inputs**: Local Python environment.
- **Outputs**: Global constants, seeds, output paths, selected compute device.
- **Constraints**: Prefer Apple MPS on M2; automatically fall back to CPU.


In [1]:
import json
import random
import re
import time
from datetime import datetime
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


SEED = 0
set_seed(SEED)

DEVICE = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
EMBED_DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

SPLIT = "validation"
N_EVAL = 200
TOP_K_LIST = [1, 3, 5]
CHUNK_SIZE = 256  # Fixed for this minimal baseline.

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
GEN_MODEL_PRIMARY = "google/flan-t5-small"
GEN_MODEL_FALLBACK = "google/flan-t5-small"

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PER_QUESTION_PATH = OUTPUT_DIR / "per_question.jsonl"
RUN_SUMMARY_PATH = OUTPUT_DIR / "run_summary.csv"

print(f"DEVICE={DEVICE}, EMBED_DEVICE={EMBED_DEVICE}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")


/Users/yuanyuan/Documents/RAG retriever/.venv-nlp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEVICE=cpu, EMBED_DEVICE=cpu
Output directory: /Users/yuanyuan/Documents/nlp_project/outputs


## 1_load_data

- **Goal**: Load SciQ and create one fixed evaluation subset.
- **Inputs**: Hugging Face dataset `sciq`.
- **Outputs**: `dataset`, `eval_indices`, `eval_subset`.
- **Constraints**: Use validation split for development; keep test split for final report only.


In [ ]:
dataset = load_dataset("sciq")

for split_name in ["train", "validation", "test"]:
    print(f"{split_name}: {len(dataset[split_name])}")


def make_fixed_subset_indices(ds_split, n: int, seed: int) -> list[int]:
    if n is None or n >= len(ds_split):
        return list(range(len(ds_split)))
    rng = np.random.default_rng(seed)
    idxs = rng.choice(len(ds_split), size=n, replace=False)
    return sorted(idxs.tolist())


eval_indices = make_fixed_subset_indices(dataset[SPLIT], N_EVAL, SEED)
eval_subset = dataset[SPLIT].select(eval_indices)

print(f"Evaluation split: {SPLIT}, n={len(eval_subset)}")
print("First 5 evaluation indices:", eval_indices[:5])


Generating test split: 100%|██████████| 1000/1000 [00:00<00:00, 540224.63 examples/s]

train: 11679
validation: 1000
test: 1000
Evaluation split: validation, n=200
First 5 evaluation indices: [2, 4, 7, 13, 14]


: 

## 2_build_corpus_index

- **Goal**: Build train-only support corpus and FAISS index.
- **Inputs**: `dataset['train']` support paragraphs.
- **Outputs**: `documents`, `support_to_id`, `embed_model`, `index`.
- **Constraints**: Train-only corpus to prevent retrieval leakage from validation/test supports.


In [ ]:
documents = []
support_to_id = {}

for ex in dataset["train"]:
    support = ex["support"]
    if support not in support_to_id:
        support_to_id[support] = len(documents)
        documents.append({"id": len(documents), "text": support})

print("Unique train supports:", len(documents))

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=EMBED_DEVICE)
doc_texts = [d["text"] for d in documents]
doc_emb = embed_model.encode(
    doc_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)

index = faiss.IndexFlatIP(doc_emb.shape[1])
index.add(doc_emb)

print("FAISS index size:", index.ntotal)


Unique train supports: 10474


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5750.67it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches:   0%|          | 0/328 [00:00<?, ?it/s]

## 3_mcq_and_prompts

- **Goal**: Define MCQ construction and prompt templates.
- **Inputs**: SciQ example (`question`, distractors, `correct_answer`) and optional retrieved context.
- **Outputs**: Prompt strings and parsed answer letters.
- **Constraints**: Use per-example seed variation (`seed + i`) to avoid fixed gold-letter artifacts.


In [ ]:
OPTION_SET = {"A", "B", "C", "D"}


def make_mcq(example: dict, seed: int):
    rnd = random.Random(seed)
    choices = [
        example["distractor1"],
        example["distractor2"],
        example["distractor3"],
        example["correct_answer"],
    ]
    rnd.shuffle(choices)
    gold_idx = choices.index(example["correct_answer"])
    gold_letter = "ABCD"[gold_idx]
    return example["question"], choices, gold_letter


def format_llm_prompt(question: str, choices: list[str]) -> str:
    return (
        "Answer the multiple-choice science question.\n"
        "Return only one capital letter: A, B, C, or D.\n"
        "Do not provide explanation.\n\n"
        f"Question: {question}\n"
        f"A) {choices[0]}\n"
        f"B) {choices[1]}\n"
        f"C) {choices[2]}\n"
        f"D) {choices[3]}\n"
    )


def format_rag_prompt(context_passages: list[str], question: str, choices: list[str]) -> str:
    context = "\n\n".join([f"[Context {i+1}]\n{p}" for i, p in enumerate(context_passages)])
    return (
        "Use the retrieved context to answer the multiple-choice science question.\n"
        "Return only one capital letter: A, B, C, or D.\n"
        "Do not provide explanation.\n\n"
        f"{context}\n\n"
        f"Question: {question}\n"
        f"A) {choices[0]}\n"
        f"B) {choices[1]}\n"
        f"C) {choices[2]}\n"
        f"D) {choices[3]}\n"
    )


def extract_option_letter(text: str | None) -> str | None:
    if text is None:
        return None
    t = text.strip()
    if not t:
        return None

    try:
        obj = json.loads(t)
        ans = str(obj.get("answer", "")).strip().upper()
        if ans in OPTION_SET:
            return ans
    except Exception:
        pass

    m = re.search(r"\b([ABCD])\b", t.upper())
    return m.group(1) if m else None


## 4_local_generator

- **Goal**: Load local text-generation model and produce answer candidates.
- **Inputs**: Prompt string.
- **Outputs**: Raw generated text, parsed option letter, token counts.
- **Constraints**: No Gemini/OpenAI API; fallback from `flan-t5-base` to `flan-t5-small` if needed.


In [ ]:
GEN_DEVICE = DEVICE


def load_generation_model(primary: str, fallback: str, device: torch.device):
    last_err = None
    for name in [primary, fallback]:
        try:
            tok = AutoTokenizer.from_pretrained(name)
            mdl = AutoModelForSeq2SeqLM.from_pretrained(name)
            mdl.to(device)
            mdl.eval()
            print(f"Loaded generator: {name} on {device}")
            return name, tok, mdl
        except Exception as e:
            last_err = e
            print(f"Failed to load {name}: {e}")
    raise RuntimeError(f"Could not load generation model. Last error: {last_err}")


GEN_MODEL_NAME, gen_tokenizer, gen_model = load_generation_model(
    GEN_MODEL_PRIMARY,
    GEN_MODEL_FALLBACK,
    GEN_DEVICE,
)


def generate_answer(prompt: str, max_new_tokens: int = 16):
    inputs = gen_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    )
    inputs = {k: v.to(GEN_DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
        )

    text = gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    input_tokens = int(inputs["input_ids"].shape[1])
    output_tokens = int(output_ids.shape[1])
    letter = extract_option_letter(text)
    return letter, text, input_tokens, output_tokens


## 5_retrieval_helpers

- **Goal**: Retrieve top-k support docs and compute retrieval-quality fields.
- **Inputs**: Question string, gold support id, top-k.
- **Outputs**: Retrieved ids/texts, retrieval hit, gold rank, MRR contribution.
- **Constraints**: Recall@k is hit-rate on gold support id; MRR uses reciprocal of gold rank (0 if miss).


In [ ]:
_query_cache = {}


def encode_query(question: str):
    if question in _query_cache:
        return _query_cache[question]
    q_emb = embed_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)
    _query_cache[question] = q_emb
    return q_emb


def retrieve_topk(question: str, k: int):
    q_emb = encode_query(question)
    t0 = time.perf_counter()
    scores, idxs = index.search(q_emb, k)
    retrieval_ms = (time.perf_counter() - t0) * 1000.0
    retrieved_ids = idxs[0].tolist()
    retrieved_texts = [documents[i]["text"] for i in retrieved_ids]
    return retrieved_ids, retrieved_texts, retrieval_ms


def get_gold_rank(gold_doc_id: int | None, retrieved_ids: list[int]) -> int | None:
    if gold_doc_id is None:
        return None
    try:
        return retrieved_ids.index(gold_doc_id) + 1
    except ValueError:
        return None


## 6_eval_loops

- **Goal**: Run LLM-only and RAG evaluations under one shared logging schema.
- **Inputs**: Fixed eval subset, mode (`llm_only` or `rag`), top-k.
- **Outputs**: Per-question records and run-level metrics.
- **Constraints**: `accuracy = correct / total`; invalid output counts as wrong.


In [ ]:
PER_QUESTION_COLUMNS = [
    "run_id",
    "question_id",
    "gold_option",
    "pred_option",
    "is_correct",
    "gold_doc_id",
    "retrieved_doc_ids",
    "gold_rank",
    "retrieval_hit",
    "input_tokens",
    "output_tokens",
    "e2e_latency_ms",
]

RUN_SUMMARY_COLUMNS = [
    "run_id",
    "chunk_size",
    "top_k",
    "embedding_model",
    "accuracy",
    "recall_at_k",
    "mrr",
    "invalid_rate",
    "retrieval_latency_ms",
    "e2e_latency_ms",
    "avg_input_tokens",
    "avg_output_tokens",
]


def evaluate_run(mode: str, top_k: int):
    assert mode in {"llm_only", "rag"}

    run_id = f"{mode}_k{top_k}_{SPLIT}_n{len(eval_subset)}_seed{SEED}_{RUN_TS}"

    rows = []
    retrieval_latencies = []
    e2e_latencies = []
    input_token_counts = []
    output_token_counts = []

    for i, (ds_idx, ex) in enumerate(zip(eval_indices, eval_subset)):
        t0 = time.perf_counter()

        question, choices, gold_letter = make_mcq(ex, seed=SEED + i)
        gold_doc_id = support_to_id.get(ex["support"])

        if mode == "rag":
            retrieved_ids, retrieved_texts, retrieval_ms = retrieve_topk(question, top_k)
            prompt = format_rag_prompt(retrieved_texts, question, choices)
            retrieval_latencies.append(retrieval_ms)
            gold_rank = get_gold_rank(gold_doc_id, retrieved_ids)
            retrieval_hit = gold_rank is not None
        else:
            retrieved_ids = []
            prompt = format_llm_prompt(question, choices)
            gold_rank = None
            retrieval_hit = False

        pred_letter, raw_text, input_tokens, output_tokens = generate_answer(prompt)

        is_correct = bool(pred_letter == gold_letter)
        e2e_ms = (time.perf_counter() - t0) * 1000.0

        input_token_counts.append(input_tokens)
        output_token_counts.append(output_tokens)
        e2e_latencies.append(e2e_ms)

        rows.append(
            {
                "run_id": run_id,
                "question_id": f"{SPLIT}_{ds_idx}",
                "gold_option": gold_letter,
                "pred_option": pred_letter,
                "is_correct": is_correct,
                "gold_doc_id": gold_doc_id,
                "retrieved_doc_ids": retrieved_ids,
                "gold_rank": gold_rank,
                "retrieval_hit": retrieval_hit,
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "e2e_latency_ms": e2e_ms,
            }
        )

    total = len(rows)
    correct = sum(1 for r in rows if r["is_correct"])
    bad = sum(1 for r in rows if r["pred_option"] is None)

    accuracy = correct / total if total else 0.0
    invalid_rate = bad / total if total else 0.0

    if mode == "rag":
        recall_at_k = sum(1 for r in rows if r["retrieval_hit"]) / total if total else 0.0
        mrr = sum((1.0 / r["gold_rank"]) if r["gold_rank"] else 0.0 for r in rows) / total if total else 0.0
        retrieval_latency_ms = float(np.mean(retrieval_latencies)) if retrieval_latencies else 0.0
    else:
        recall_at_k = 0.0
        mrr = 0.0
        retrieval_latency_ms = 0.0

    summary = {
        "run_id": run_id,
        "chunk_size": CHUNK_SIZE,
        "top_k": top_k,
        "embedding_model": EMBED_MODEL_NAME,
        "accuracy": accuracy,
        "recall_at_k": recall_at_k,
        "mrr": mrr,
        "invalid_rate": invalid_rate,
        "retrieval_latency_ms": retrieval_latency_ms,
        "e2e_latency_ms": float(np.mean(e2e_latencies)) if e2e_latencies else 0.0,
        "avg_input_tokens": float(np.mean(input_token_counts)) if input_token_counts else 0.0,
        "avg_output_tokens": float(np.mean(output_token_counts)) if output_token_counts else 0.0,
    }

    rows_df = pd.DataFrame(rows, columns=PER_QUESTION_COLUMNS)
    summary_df = pd.DataFrame([summary], columns=RUN_SUMMARY_COLUMNS)
    return rows_df, summary_df


## 7_run_experiments

- **Goal**: Execute one LLM-only run and three RAG runs (`k=1/3/5`).
- **Inputs**: Fixed subset (`eval_indices`) and model/index objects.
- **Outputs**: `per_question_df`, `run_summary_df`.
- **Constraints**: All runs must share the same split, subset size, and seed.


In [ ]:
all_per_question = []
all_summaries = []

# Pure LLM baseline
rows_df, summary_df = evaluate_run(mode="llm_only", top_k=0)
all_per_question.append(rows_df)
all_summaries.append(summary_df)

# RAG runs with different top-k
for k in TOP_K_LIST:
    rows_df, summary_df = evaluate_run(mode="rag", top_k=k)
    all_per_question.append(rows_df)
    all_summaries.append(summary_df)

per_question_df = pd.concat(all_per_question, ignore_index=True)
run_summary_df = pd.concat(all_summaries, ignore_index=True)

display(run_summary_df.sort_values(by=["top_k"]).reset_index(drop=True))


## 8_save_outputs

- **Goal**: Persist reproducible outputs for downstream analysis.
- **Inputs**: In-memory per-question and run-summary DataFrames.
- **Outputs**: `outputs/per_question.jsonl`, `outputs/run_summary.csv`.
- **Constraints**: Field names must match the agreed public schemas.


In [ ]:
# Save per-question records as JSONL
with PER_QUESTION_PATH.open("w", encoding="utf-8") as f:
    for row in per_question_df[PER_QUESTION_COLUMNS].to_dict(orient="records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# Save run-level summary as CSV
run_summary_df[RUN_SUMMARY_COLUMNS].to_csv(RUN_SUMMARY_PATH, index=False)

print(f"Saved per-question records: {PER_QUESTION_PATH.resolve()}")
print(f"Saved run summary: {RUN_SUMMARY_PATH.resolve()}")


## 9_quick_checks

- **Goal**: Run minimum sanity checks before reporting any number.
- **Inputs**: Parser, evaluation outputs, and subset metadata.
- **Outputs**: Pass/fail messages for parser, shuffle sanity, metric consistency, and fairness.
- **Constraints**: Fail fast on consistency issues; do not trust metrics if checks fail.


In [ ]:
# 1) Output parser sanity checks
parser_cases = {
    '{"answer": "A"}': "A",
    "The answer is B": "B",
    "": None,
    "No valid option": None,
}

for txt, expected in parser_cases.items():
    got = extract_option_letter(txt)
    print(f"Parser case: {txt!r} -> {got}, expected={expected}")

# 2) MCQ shuffle sanity check: gold letters should not collapse to one constant letter
first_n = min(50, len(eval_subset))
gold_letters = []
for i in range(first_n):
    _, _, g = make_mcq(eval_subset[i], seed=SEED + i)
    gold_letters.append(g)
print("Unique gold letters in first 50 examples:", sorted(set(gold_letters)))
assert len(set(gold_letters)) > 1, "Gold letters collapsed to one value; check seed usage."

# 3) Retrieval hit/rank controlled logic check
_example_gold = 7
_example_retrieved = [3, 7, 9]
assert get_gold_rank(_example_gold, _example_retrieved) == 2
assert get_gold_rank(8, _example_retrieved) is None
print("Retrieval rank helper checks passed.")

# 4) Metric consistency check from per-question rows
for run_id, g in per_question_df.groupby("run_id"):
    recomputed_acc = g["is_correct"].mean()
    recomputed_invalid = g["pred_option"].isna().mean()
    row = run_summary_df.loc[run_summary_df["run_id"] == run_id].iloc[0]
    assert abs(recomputed_acc - row["accuracy"]) < 1e-12, f"Accuracy mismatch for {run_id}"
    assert abs(recomputed_invalid - row["invalid_rate"]) < 1e-12, f"Invalid rate mismatch for {run_id}"
print("Metric consistency checks passed.")

# 5) Fairness check: all runs must use exactly the same question ids
qid_sets = per_question_df.groupby("run_id")["question_id"].apply(lambda s: set(s.tolist()))
base_qids = qid_sets.iloc[0]
for rid, qset in qid_sets.items():
    assert qset == base_qids, f"Question-id mismatch found in run {rid}"
print("Fairness check passed: all runs evaluated on identical question IDs.")
